In [ ]:
import os, sys, json, glob
from pathlib import Path
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from PIL import Image

load_dotenv(Path('..') / '.env')
sys.path.insert(0, str(Path('..')))

from lib.client import VISION_MODEL
from lib.images import resize_image
from lib.categorize import categorize_item

print(f'VISION_MODEL: {VISION_MODEL}')
print(f'API key set: {bool(os.getenv("GEMINI_API_KEY"))}')
Path('../outputs').mkdir(exist_ok=True)

## Step 1 — Load 2–4 photos of the same item

Put your multi-angle photos in `inputs/multi_angle/` or point `PHOTO_PATHS` at any list of paths.

Example: flat-lay + worn photo of the same jacket.

In [ ]:
# Option A: folder of photos for one item
MULTI_ANGLE_DIR = Path('../inputs/multi_angle')
if MULTI_ANGLE_DIR.exists():
    PHOTO_PATHS = sorted(MULTI_ANGLE_DIR.glob('*.jpg')) + sorted(MULTI_ANGLE_DIR.glob('*.png'))
else:
    # Option B: pick specific DB items manually
    db_items = sorted(glob.glob(os.path.expanduser('~/.closet/images/items/*.jpg')))
    PHOTO_PATHS = db_items[:2]  # ← set to 2-4 photos of the SAME item
    print('multi_angle/ dir not found — using first 2 DB items as placeholder')
    print('Create inputs/multi_angle/ and add 2-4 photos of one item for a real test')

print(f'Loaded {len(PHOTO_PATHS)} photo(s):')
for p in PHOTO_PATHS:
    print(f'  {p}')

imgs = [resize_image(Image.open(p)) for p in PHOTO_PATHS]

cols = len(imgs)
fig, axes = plt.subplots(1, cols, figsize=(cols * 4, 5))
if cols == 1:
    axes = [axes]
for ax, img, p in zip(axes, imgs, PHOTO_PATHS):
    ax.imshow(img)
    ax.set_title(Path(p).name, fontsize=8)
    ax.axis('off')
plt.suptitle('Input photos (same item, multiple angles)', fontsize=12)
plt.tight_layout()
plt.show()

## Step 2 — [API] Categorize each photo independently

Runs `categorize_item()` on each angle. Shows where per-angle results agree or diverge.

In [ ]:
print('Categorizing each angle...')
per_angle = []
for i, img in enumerate(imgs):
    print(f'  Angle {i+1}/{len(imgs)}...')
    cl = categorize_item(img)
    per_angle.append(cl)
    print(f'    → {cl.get("category")} / {cl.get("subcategory")} ({cl.get("primary_color")})')

print('\nPer-angle classifications:')
for i, cl in enumerate(per_angle):
    print(f'\nAngle {i+1}:')
    print(json.dumps(cl, indent=2))

## Step 3 — [API] Merge classifications into one canonical item

Send all per-angle results + all images to Gemini with a merge prompt. Ask it to produce one definitive ItemClassification that reconciles disagreements across angles.

In [ ]:
from lib.client import get_client, with_retry
from lib.parse import parse_gemini_json
from lib.images import image_to_bytes

MERGE_PROMPT = """You are given {n} photos of the SAME clothing item from different angles, plus per-photo AI classifications.
Produce ONE canonical classification that best describes the item across all views.
Prefer the most specific and accurate values — if one angle shows the label clearly, trust it.
Return ONLY valid JSON matching this schema:
{{
  "category": "...",
  "subcategory": "...",
  "primary_color": "...",
  "colors": [...],
  "material": "...",
  "season": [...],
  "ai_description": "...",
  "suggested_tags": [...],
  "wear_context": null
}}

Per-angle classifications for reference:
{classifications}
"""

def merge_classifications(imgs, per_angle):
    parts = []
    for img in imgs:
        parts.append({"inline_data": {"mime_type": "image/jpeg", "data": image_to_bytes(img)}})
    parts.append({"text": MERGE_PROMPT.format(
        n=len(imgs),
        classifications=json.dumps(per_angle, indent=2)
    )})

    def _call():
        client = get_client()
        response = client.models.generate_content(
            model=VISION_MODEL,
            contents=[{"role": "user", "parts": parts}],
        )
        result = parse_gemini_json(response.text)
        if isinstance(result, list):
            return result[0]
        return result

    return with_retry(_call, label="[merge]")

print('Merging classifications...')
merged = merge_classifications(imgs, per_angle)
print('Merged classification:')
print(json.dumps(merged, indent=2))

## Step 4 — Compare: single-angle vs merged

Side-by-side of what angle 1 produced alone vs the merged result. Key question: does the merged result add info that no single angle had?

In [ ]:
fields = ['category', 'subcategory', 'primary_color', 'material', 'season', 'suggested_tags', 'ai_description']

print(f'{"Field":<20} {"Angle 1":<35} {"Merged":<35}')
print('-' * 90)
for f in fields:
    v1 = str(per_angle[0].get(f, '—'))
    vm = str(merged.get(f, '—'))
    marker = '★' if v1 != vm else ' '
    print(f'{marker} {f:<19} {v1[:33]:<35} {vm[:33]:<35}')
print('\n★ = merged differs from angle-1 alone')

## Step 5 — [Optional] Generate hero thumbnail

Requires `IMAGE_GEN_MODEL` (gemini-3.1-flash-image-preview). Generates a clean flat-lay product shot from the merged description.

In [ ]:
import base64, io
from lib.client import get_client

IMAGE_GEN_MODEL = os.getenv('IMAGE_GEN_MODEL', 'gemini-3.1-flash-image-preview')

hero_prompt = (
    f"Professional product photography: {merged.get('ai_description', '')}. "
    f"Clean white background, flat-lay style, "
    f"{merged.get('primary_color', '')} {merged.get('subcategory', '')}. "
    "No people, no props, just the garment. High resolution, studio lighting."
)

print(f'Generating hero image with {IMAGE_GEN_MODEL}...')
print(f'Prompt: {hero_prompt[:120]}...')

try:
    client = get_client()
    response = client.models.generate_content(
        model=IMAGE_GEN_MODEL,
        contents=hero_prompt,
        config={"response_modalities": ["IMAGE", "TEXT"]},
    )
    for part in response.candidates[0].content.parts:
        if hasattr(part, 'inline_data') and part.inline_data:
            img_data = base64.b64decode(part.inline_data.data)
            hero_img = Image.open(io.BytesIO(img_data))
            hero_img.save('../outputs/03_hero_thumbnail.png')
            plt.figure(figsize=(6, 6))
            plt.imshow(hero_img)
            plt.axis('off')
            plt.title('Generated hero thumbnail')
            plt.tight_layout()
            plt.show()
            print('Saved → outputs/03_hero_thumbnail.png')
            break
    else:
        print('No image in response — model may not support image generation')
        print(response.text)
except Exception as e:
    print(f'Hero generation failed: {e}')
    print('This is expected if IMAGE_GEN_MODEL is not available on your quota tier.')